# Clase 4 — Churn en GCP: de los datos al scoring batch

Esta notebook es el lab de la clase 4. La corremos en **Cloud Shell**, con el repo del curso clonado.
La idea es simple: cada paso es **un concepto, una celda que apretás, y un lugar de la consola donde
ver el recurso que apareció**.

**Antes de arrancar necesitás:**
- Tu cuenta de GCP activa y un proyecto con billing (eso lo hicimos en la Parte 0).
- Cloud Shell abierto y este repo clonado.
- Estar parado en la carpeta `notebooks/` del repo clonado.

Caso guía: **Customer Churn**. El usuario es el equipo de **CRM**. Queremos una lista de a quién
llamar primero, ordenada por `priority_score = churn_probability × MonthlyCharges`.

## Paso 0 — Dónde estás parado

Lo primero en la nube es siempre lo mismo: confirmar **en qué proyecto** vas a trabajar. Todo lo que
crees hoy (bucket, modelo, jobs) vive dentro de ese proyecto. Fijate arriba a la izquierda en la
consola: ese nombre tiene que coincidir con lo que imprime esta celda.

In [ ]:
# En Cloud Shell, gcloud ya está autenticado con tu cuenta.
PROJECT_ID = !gcloud config get-value project 2>/dev/null
PROJECT_ID = PROJECT_ID[0].strip()
REGION = "us-central1"
BUCKET = f"{PROJECT_ID}-mlops-2026-churn"

print("Proyecto:", PROJECT_ID)
print("Región:  ", REGION)
print("Bucket:  ", BUCKET)

## Paso 1 — Prender las APIs

Un proyecto de GCP nace con casi todo **apagado**. Antes de usar un servicio hay que **habilitar su
API**. Prendemos las tres que necesitamos: Vertex AI (`aiplatform`), Cloud Storage (`storage`) y
Artifact Registry (para las clases que vienen).

**Andá a ver:** en la consola, *APIs y servicios → APIs habilitadas*. Van a aparecer estas tres.

In [ ]:
!gcloud services enable \
  aiplatform.googleapis.com \
  storage.googleapis.com \
  artifactregistry.googleapis.com

## Paso 2 — El dato a la nube

El dato deja de vivir en una compu y pasa a **Cloud Storage**, el almacenamiento de objetos de GCP
(el equivalente al S3 de AWS). Primero verificamos que el CSV canónico esté sano (forma y checksum),
después creamos un **bucket** y subimos el archivo.

**Andá a ver:** en la consola, *Cloud Storage → Buckets → tu bucket → `raw/`*. Ahí está tu CSV.

In [ ]:
# Verificamos forma y checksum del dataset canónico (7043 filas, 21 columnas).
!python ../scripts/verify_dataset.py

# Creamos el bucket (si ya existe, no pasa nada) y subimos el CSV.
!gsutil mb -l {REGION} gs://{BUCKET} || true
!gsutil cp ../data/raw/Telco-Customer-Churn.csv gs://{BUCKET}/raw/
!gsutil ls -l gs://{BUCKET}/raw/

## Paso 3 — Entrenar

Acá pasás de **datos** a **modelo**. Reutilizamos el pipeline del caso guía (imputación, escalado,
one-hot y una regresión logística balanceada). Entrena en **segundos**: es el camino *a mano*, rápido
y reproducible. Vertex AutoML haría algo parecido, pero tardaría horas; lo miramos al final.

Cuando termine, leé el **AUC** y el **recall**: el recall es qué proporción de los que se van de
verdad estás agarrando. Para CRM, un recall alto significa **no dejar pasar** clientes en riesgo.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score

from scripts.train_baseline import build_pipeline, load_training_data

DATA = Path("../data/raw/Telco-Customer-Churn.csv")
df = pd.read_csv(DATA)                      # crudo, lo usamos después para customerID
X, y = load_training_data(DATA)             # features limpias + etiqueta 0/1

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = build_pipeline()
pipeline.fit(X_tr, y_tr)

proba = pipeline.predict_proba(X_te)[:, 1]
pred = proba >= 0.7

{
    "roc_auc": round(roc_auc_score(y_te, proba), 3),
    "precision@0.7": round(precision_score(y_te, pred), 3),
    "recall@0.7": round(recall_score(y_te, pred), 3),
}

## Paso 4 — Registrar y versionar el modelo

Un modelo sin registrar es un archivo perdido. Le ponemos una **versión** con fecha y guardamos el
artefacto más sus métricas. Después lo subimos a Cloud Storage: así el modelo también **vive en la
nube**, con su linaje.

> Nota: el **Model Registry gestionado** de Vertex, para un modelo propio, pide además un contenedor
> de serving. Ese pozo lo dejamos para más adelante. Hoy el "registro" es el artefacto **versionado
> en Cloud Storage**, y el Registry real lo vas a ver en la demo de AutoML.

In [ ]:
import json
import joblib
from datetime import datetime, timezone

version = datetime.now(timezone.utc).strftime("churn-baseline-%Y%m%dT%H%M%SZ")
Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(
    {"pipeline": pipeline, "metadata": {"model_version": version}},
    "../models/churn-baseline.joblib",
)
json.dump(
    {"model_version": version, "roc_auc": float(roc_auc_score(y_te, proba))},
    open("../models/churn-baseline-metrics.json", "w"),
    indent=2,
)
print("Modelo versionado:", version)

In [ ]:
# Cloud Shell: subimos el modelo versionado a la nube.
!gsutil cp ../models/churn-baseline.joblib gs://{BUCKET}/models/
!gsutil cp ../models/churn-baseline-metrics.json gs://{BUCKET}/models/
!gsutil ls -l gs://{BUCKET}/models/

## Paso 5 — Scoring batch

Esta es la inferencia **batch**: en vez de responder de a un cliente, scoreamos a **todos** de una
(acá usamos el conjunto de test como si fueran los clientes activos del mes) y armamos el ranking.

La prioridad no es solo la probabilidad de baja: es `churn_probability × MonthlyCharges`. Así CRM
llama primero a quien **se va y factura mucho**.

**Andá a ver:** después de subir el resultado, aparece en *Cloud Storage → `scored/`*.

In [ ]:
scored = X_te.copy()
scored["customerID"] = df.loc[X_te.index, "customerID"]
scored["churn_probability"] = proba
scored["priority_score"] = scored["churn_probability"] * scored["MonthlyCharges"]

ranking = scored.sort_values("priority_score", ascending=False)

# Guardamos y subimos el resultado del batch.
Path("../data/scored").mkdir(parents=True, exist_ok=True)
ranking.to_csv("../data/scored/batch-scored.csv", index=False)

ranking[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

In [ ]:
# Cloud Shell: el resultado del batch también va a la nube.
!gsutil cp ../data/scored/batch-scored.csv gs://{BUCKET}/scored/
!gsutil ls -l gs://{BUCKET}/scored/

## Paso 6 — De score a decisión

El número no vale solo: vale porque **ordena una acción**. Aplicamos el umbral del caso: los clientes
con `churn_probability >= 0.7` entran a la **cola de retención**; al resto lo monitoreamos.

Esta es la lista concreta que CRM abre el lunes a la mañana.

In [ ]:
umbral = 0.7
cola = ranking[ranking["churn_probability"] >= umbral]

print(f"Clientes en cola de retención (P >= {umbral}): {len(cola)} de {len(ranking)}")
cola[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

## Paso 7 — Costos y limpieza

Regla de oro de la nube: **lo que prendés, cuesta**. Hoy tuvimos suerte: no desplegamos ningún
endpoint ni dejamos ninguna máquina prendida, así que no hay costo de serving corriendo. Lo único que
queda es el bucket con unos pocos MB, que entra holgado en el free tier.

Si querés dejar todo limpio, podés borrar el bucket de prueba. **Cuidado**: esto borra el dato, el
modelo y el resultado que subiste.

In [ ]:
# Descomentá solo si querés borrar TODO lo que subimos hoy:
# !gsutil -m rm -r gs://{BUCKET}
print("Nada quedó prendido cobrando. El bucket es lo único que persiste.")

## Bonus — AutoML, para que lo veas (concepto)

Lo que hicimos a mano en segundos, Vertex lo puede hacer **solo**: le das el dataset tabular, le
decís que la columna a predecir es `Churn`, y prueba modelos por vos. El resultado es parecido; el
costo es **tiempo y crédito** (el entrenamiento tarda alrededor de dos horas), por eso no lo corremos
en vivo.

El flujo gestionado, para que lo ubiques, es este (lo vemos con capturas ya cocinadas en clase):

```bash
# 1) Crear un dataset tabular en Vertex desde el CSV en Cloud Storage.
# 2) Entrenar una clasificación binaria con target = Churn.
# 3) Revisar las métricas de evaluación que Vertex calcula solo.
# 4) Registrar el modelo en el Model Registry.
# 5) Correr un batch prediction sobre los clientes activos.
```

Los comandos parametrizados están en `../gcp/runbook.md`, sección *Clase 4*.

## Cierre

Recorriste el ciclo entero de una punta: **dato → modelo → registro → batch → decisión**. De un CSV
salió una lista priorizada para CRM.

En la **clase 5** damos vuelta la pregunta: ¿cómo hace *otro sistema* para pedirle el score a este
modelo en el momento, sin abrir esta notebook? Ese mismo modelo, detrás de una **API** con contrato
estable.

**Tarea:** corré esta misma notebook con el **dato de tu TFI**. Con que entrenes algo y veas un
score, alcanza. Anotá qué te rompió.